<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/04_gpu_monte_carlo_calibration.ipynb)


# colab — GPU Monte Carlo Calibration

## _From Option Pricing Simulation to Model Fitting on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Calibrate volatility to a synthetic option surface by running
  many Monte Carlo pricing evaluations on CPU and GPU.
- **Hardware**: Designed for Google Colab (T4 default). GPU is
  auto-detected, with a CPU fallback.
- **Data**: Fully synthetic option prices, no market data download or
  credentials required.
- **Approach**: Generate a synthetic option surface, search over volatility
  candidates, and compare CPU vs GPU calibration time.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Check GPU and import NumPy, PyTorch, Matplotlib.
2. **Market**: Create a synthetic option surface from known parameters.
3. **CPU Search**: Price the surface for many volatility candidates.
4. **GPU Search**: Vectorise the same calibration workload on CUDA.
5. **Results**: Compare estimated parameters, errors, and timings.
6. **Visuals**: Plot the loss curve and fitted option surface.
7. **Exercises**: Extend the calibration experiment.

### Environment Setup

We verify the GPU and install the minimal package set needed for numerical
simulation, tabular summaries, and plotting.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install matplotlib numpy pandas torch

In [ ]:
#@title Imports and helpers
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

def bench(label, fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f} s")
    return result, elapsed

### Synthetic Option Market

We create a small option surface from the Black-Scholes-Merton formula.
In practice, these prices would come from observed market quotes. Here,
synthetic data keeps the notebook deterministic and zero-upload.

In [ ]:
#@title Market grid and true parameters
S0 = 100.0
r = 0.03
true_sigma = 0.24
seed = 42

strikes = torch.tensor([80, 90, 100, 110, 120], dtype=torch.float32)
maturities = torch.tensor([0.25, 0.5, 1.0, 2.0], dtype=torch.float32)
K_grid, T_grid = torch.meshgrid(strikes, maturities, indexing="ij")
K_flat = K_grid.flatten()
T_flat = T_grid.flatten()

print(f"Options: {len(K_flat)}")
print(f"True volatility: {true_sigma:.2%}")

In [ ]:
#@title Generate synthetic market prices
normal = torch.distributions.Normal(0.0, 1.0)

def bsm_call_torch(S0, K, T, r, sigma):
    sqrt_T = torch.sqrt(T)
    d1 = (torch.log(S0 / K) + (r + 0.5 * sigma ** 2) * T)
    d1 = d1 / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    price = S0 * normal.cdf(d1)
    price = price - K * torch.exp(-r * T) * normal.cdf(d2)
    return price

torch.manual_seed(seed)
clean_prices = bsm_call_torch(S0, K_flat, T_flat, r, true_sigma)
noise = 0.02 * torch.randn_like(clean_prices)
market_prices = torch.clamp(clean_prices + noise, min=0.01)

market_df = pd.DataFrame({
    "strike": K_flat.numpy(),
    "maturity": T_flat.numpy(),
    "market_price": market_prices.numpy(),
})
market_df.head()

In [ ]:
#@title Plot synthetic market surface
surface = market_prices.reshape(len(strikes), len(maturities)).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(surface, aspect="auto", origin="lower")
ax.set_xticks(range(len(maturities)))
ax.set_xticklabels([f"{x:.2f}" for x in maturities.numpy()])
ax.set_yticks(range(len(strikes)))
ax.set_yticklabels([f"{x:.0f}" for x in strikes.numpy()])
ax.set_xlabel("Maturity")
ax.set_ylabel("Strike")
ax.set_title("Synthetic Market Call Prices")
fig.colorbar(im, ax=ax, label="Price")
plt.show()

### Calibration as a Compute Problem

Calibration means choosing model parameters that minimize pricing errors.
Even this one-parameter volatility search requires many option pricing
runs. More realistic models need more parameters and many more searches.

In [ ]:
#@title Calibration setup
sigma_candidates = np.linspace(0.10, 0.45, 71).astype(np.float32)
n_paths = 40_000

K_np = K_flat.numpy().astype(np.float32)
T_np = T_flat.numpy().astype(np.float32)
market_np = market_prices.numpy().astype(np.float32)
rng = np.random.default_rng(seed)
z_np = rng.standard_normal((len(K_np), n_paths)).astype(np.float32)

print(f"Candidates: {len(sigma_candidates)}")
print(f"Paths per option: {n_paths:,}")
print(f"Pricing evaluations: {len(sigma_candidates) * len(K_np):,}")

In [ ]:
#@title CPU calibration with NumPy
def calibrate_cpu(sigmas, z, K, T, market):
    losses = []
    prices_by_sigma = []
    for sigma in sigmas:
        drift = (r - 0.5 * sigma ** 2) * T[:, None]
        diffusion = sigma * np.sqrt(T[:, None]) * z
        ST = S0 * np.exp(drift + diffusion)
        payoff = np.maximum(ST - K[:, None], 0.0)
        prices = np.exp(-r * T) * payoff.mean(axis=1)
        loss = np.mean((prices - market) ** 2)
        losses.append(loss)
        prices_by_sigma.append(prices)
    idx = int(np.argmin(losses))
    return idx, np.array(losses), np.array(prices_by_sigma)

(idx_cpu, losses_cpu, prices_cpu), t_cpu = bench(
    "CPU calibration", calibrate_cpu,
    sigma_candidates, z_np, K_np, T_np, market_np,
)
sigma_cpu = float(sigma_candidates[idx_cpu])
print(f"CPU estimate: {sigma_cpu:.2%}")

In [ ]:
#@title GPU calibration with PyTorch CUDA
def calibrate_gpu(sigmas, z, K, T, market, device):
    sig = torch.tensor(sigmas, device=device).view(-1, 1, 1)
    z_t = torch.tensor(z, device=device).unsqueeze(0)
    K_t = torch.tensor(K, device=device).view(1, -1, 1)
    T_t = torch.tensor(T, device=device).view(1, -1, 1)
    m_t = torch.tensor(market, device=device).view(1, -1)

    drift = (r - 0.5 * sig ** 2) * T_t
    diffusion = sig * torch.sqrt(T_t) * z_t
    ST = S0 * torch.exp(drift + diffusion)
    payoff = torch.clamp(ST - K_t, min=0.0)
    prices = torch.exp(-r * T_t.squeeze(-1)) * payoff.mean(dim=2)
    losses = torch.mean((prices - m_t) ** 2, dim=1)
    if device == "cuda":
        torch.cuda.synchronize()
    idx = int(torch.argmin(losses).item())
    return idx, losses.cpu().numpy(), prices.cpu().numpy()

(idx_gpu, losses_gpu, prices_gpu), t_gpu = bench(
    "GPU calibration", calibrate_gpu,
    sigma_candidates, z_np, K_np, T_np, market_np, device,
)
sigma_gpu = float(sigma_candidates[idx_gpu])
print(f"GPU estimate: {sigma_gpu:.2%}")

### Calibration Results

The best candidate should be close to the volatility used to generate the
synthetic market surface. Monte Carlo noise and quote noise explain small
differences.

In [ ]:
#@title Summary table
summary = pd.DataFrame([
    {
        "method": "CPU / NumPy",
        "sigma_est": sigma_cpu,
        "loss": float(losses_cpu[idx_cpu]),
        "time_s": t_cpu,
    },
    {
        "method": f"GPU / PyTorch ({device})",
        "sigma_est": sigma_gpu,
        "loss": float(losses_gpu[idx_gpu]),
        "time_s": t_gpu,
    },
])
summary["speedup_vs_cpu"] = t_cpu / summary["time_s"]
summary["sigma_est"] = summary["sigma_est"].map(lambda x: f"{x:.2%}")
summary

In [ ]:
#@title Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sigma_candidates, losses_cpu, label="CPU loss")
ax.plot(sigma_candidates, losses_gpu, "--", label="GPU loss")
ax.axvline(true_sigma, color="black", linestyle=":",
           label="true sigma")
ax.axvline(sigma_gpu, color="tab:green", linestyle="--",
           label="GPU estimate")
ax.set_xlabel("Volatility candidate")
ax.set_ylabel("Mean squared pricing error")
ax.set_title("Calibration Loss Curve")
ax.legend()
plt.show()

In [ ]:
#@title Market vs fitted prices
best_prices = prices_gpu[idx_gpu]
fit_df = market_df.copy()
fit_df["fitted_price"] = best_prices
fit_df["error"] = fit_df["fitted_price"] - fit_df["market_price"]
fit_df.head(10)

In [ ]:
#@title CPU vs GPU timing
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(summary["method"], summary["time_s"],
       color=["tab:blue", "tab:green"])
ax.set_ylabel("Seconds")
ax.set_title("Calibration Runtime")
for i, value in enumerate(summary["time_s"]):
    ax.text(i, value, f"{value:.2f}s", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.show()

print(f"GPU speed-up vs CPU: {t_cpu / t_gpu:.1f}x")

### What Changes in Real Calibration?

Real desks calibrate several parameters, use bid/ask spreads, weight liquid
options more heavily, and often combine global and local optimization. The
compute lesson remains: repeated pricing calls make calibration a natural
candidate for GPU acceleration.

### Exercises

1. **More paths**: Increase `n_paths` and observe runtime and stability.
2. **Finer grid**: Increase the number of volatility candidates.
3. **More options**: Add strikes and maturities to the option surface.
4. **Noise test**: Increase quote noise and inspect estimation accuracy.
5. **Richer model**: Add a second parameter and run a two-dimensional grid.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
